In [ ]:
!pwd
%cd /data/wangshunyi/mmdetection/
!pwd

In [ ]:
# COCO数据集转换
import json

# 加载原始标注文件
with open("/data/wangshunyi/mmdetection/data/DENTEX/training_data/quadrant_enumeration/train_quadrant_enumeration.json", 'r') as f:
    coco_data = json.load(f)

# 定义新的 categories 列表
new_categories = []
for quad_id in range(4):
    base = [10, 20, 30, 40][quad_id]  # 对应象限的 FDI 基数
    for tooth_id in range(8):
        fdi_number = base + tooth_id + 1
        new_categories.append({"id": quad_id * 8 + tooth_id, "name": str(fdi_number), "supercategory": "tooth"})

# 更新每个标注的 category_id
for ann in coco_data['annotations']:
    new_id = ann['category_id_1'] * 8 + ann['category_id_2']
    ann['category_id'] = new_id
    del ann['category_id_1']
    del ann['category_id_2']

# 替换 categories 字段
coco_data['categories'] = new_categories
del coco_data['categories_1']
del coco_data['categories_2']

# 保存新的标注文件
with open('/data/wangshunyi/mmdetection/data/DENTEX/training_data/quadrant_enumeration/new_train_quadrant_enumeration.json', 'w') as f:
    json.dump(coco_data, f, indent=2)

In [ ]:
# 数据集划分
import os
import json
import numpy as np
import shutil
 
# 数据集路径
dataset_root = "/data/wangshunyi/mmdetection/data/DENTEX/"
images_folder = os.path.join(dataset_root, "training_data/quadrant_enumeration/xrays")
annotations_path = os.path.join(dataset_root, "training_data/quadrant_enumeration/new_train_quadrant_enumeration.json")
 
# 输出路径
output_root = os.path.join(dataset_root, "new_enumeration_data")
os.makedirs(output_root, exist_ok=True)
 
# 读取annotations.json文件
with open(annotations_path, "r") as f:
    annotations_data = json.load(f)
 
# 提取images, annotations, categories
images = annotations_data["images"]
annotations = annotations_data["annotations"]
categories = annotations_data["categories"]
 
# 随机打乱数据
np.random.shuffle(images)
 
# 训练集，验证集，测试集比例
train_ratio, val_ratio, test_ratio = 0.7, 0.1, 0.2
 
# 计算训练集，验证集，测试集的大小
num_images = len(images)
num_train = int(num_images * train_ratio)
num_val = int(num_images * val_ratio)
 
# 划分数据集
train_images = images[:num_train]
val_images = images[num_train:num_train + num_val]
test_images = images[num_train + num_val:]
 
# 分别为训练集、验证集和测试集创建子文件夹
train_folder = os.path.join(output_root, "train")
val_folder = os.path.join(output_root, "val")
test_folder = os.path.join(output_root, "test")
os.makedirs(train_folder, exist_ok=True)
os.makedirs(val_folder, exist_ok=True)
os.makedirs(test_folder, exist_ok=True)
 
# 将图片文件复制到相应的子文件夹
for img in train_images:
    shutil.copy(os.path.join(images_folder, img["file_name"]), os.path.join(train_folder, img["file_name"]))
 
for img in val_images:
    shutil.copy(os.path.join(images_folder, img["file_name"]), os.path.join(val_folder, img["file_name"]))
 
for img in test_images:
    shutil.copy(os.path.join(images_folder, img["file_name"]), os.path.join(test_folder, img["file_name"]))
 
# 根据图片id分配annotations
def filter_annotations(annotations, image_ids):
    return [ann for ann in annotations if ann["image_id"] in image_ids]
 
train_ann = filter_annotations(annotations, [img["id"] for img in train_images])
val_ann = filter_annotations(annotations, [img["id"] for img in val_images])
test_ann = filter_annotations(annotations, [img["id"] for img in test_images])
 
# 生成train.json, val.json, test.json
train_json = {"images": train_images, "annotations": train_ann, "categories":categories}
val_json = {"images": val_images, "annotations": val_ann, "categories": categories}
test_json = {"images": test_images, "annotations": test_ann, "categories": categories}
 
with open(os.path.join(output_root, "train.json"), "w") as f:
    json.dump(train_json, f)
 
with open(os.path.join(output_root, "val.json"), "w") as f:
    json.dump(val_json, f)
 
with open(os.path.join(output_root, "test.json"), "w") as f:
    json.dump(test_json, f)
 
print("数据集划分完成！")

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt

# 设置文件路径
ann_file = '/data/wangshunyi/mmdetection/data/DENTEX/new_enumeration_data/train.json'  # 替换为您的标注文件路径
img_dir = '/data/wangshunyi/mmdetection/data/DENTEX/new_enumeration_data/train/'    # 替换为您的图片文件夹路径

# 加载标注文件
with open(ann_file, 'r') as f:
    coco_data = json.load(f)

# 获取第一张图片的信息
img_info = coco_data['images'][0]
print("images:",img_info)
img_id = img_info['id']
img_file = img_dir + img_info['file_name']

# 加载图片
img = cv2.imread(img_file)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # 转换为 RGB 格式以便显示

# 获取该图片的标注
annotations = [ann for ann in coco_data['annotations'] if ann['image_id'] == img_id]

# 获取类别信息（FDI 编号）
categories = {cat['id']: cat['name'] for cat in coco_data['categories']}

# 绘制边界框和标签
for ann in annotations:
    bbox = ann['bbox']  # 边界框格式：[x, y, width, height]
    category_id = ann['category_id']
    label = categories[category_id]  # 获取 FDI 编号作为标签

    # 边界框坐标
    x, y, w, h = map(int, bbox)
    # 绘制矩形（绿色，线宽 2）
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    # 添加标签（绿色，字体大小 0.9）
    cv2.putText(img, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

# 显示图片
plt.figure(figsize=(100, 80))  # 设置显示大小
plt.imshow(img)
plt.axis('off')  # 隐藏坐标轴
plt.show()

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt

# 设置文件路径
ann_file = '/data/wangshunyi/mmdetection/data/DENTEX/training_data/quadrant_enumeration/train_quadrant_enumeration.json'  # 替换为您的标注文件路径
img_dir = '/data/wangshunyi/mmdetection/data/DENTEX/training_data/quadrant_enumeration/xrays/'    # 替换为您的图片文件夹路径

# 加载标注文件
with open(ann_file, 'r') as f:
    coco_data = json.load(f)

# 获取第一张图片的信息
img_info = coco_data['images'][3]
img_id = img_info['id']
img_file = img_dir + img_info['file_name']

# 加载图片
img = cv2.imread(img_file)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # 转换为 RGB 格式以便显示

# 获取该图片的标注
annotations = [ann for ann in coco_data['annotations'] if ann['image_id'] == img_id]

# 获取类别信息（FDI 编号）
categories = {cat['id']: cat['name'] for cat in coco_data['categories_1']}

# 绘制边界框和标签
for ann in annotations:
    bbox = ann['bbox']  # 边界框格式：[x, y, width, height]
    category_id = ann['category_id_1']
    label = categories[category_id]  # 获取 FDI 编号作为标签

    # 边界框坐标
    x, y, w, h = map(int, bbox)
    # 绘制矩形（绿色，线宽 2）
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    # 添加标签（绿色，字体大小 0.9）
    cv2.putText(img, str(label), (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

# 显示图片
plt.figure(figsize=(100, 100))  # 设置显示大小
plt.imshow(img)
plt.axis('off')  # 隐藏坐标轴
plt.show()

In [ ]:
import json
import numpy as np

# 加载COCO格式标注文件
with open('/data/wangshunyi/mmdetection/data/DENTEX/new_enumeration_data/train.json', 'r') as f:
    data = json.load(f)

# 提取所有牙齿标注的宽高比（width/height）
ratios = []
for ann in data['annotations']:
    # COCO格式bbox为[x_min, y_min, width, height]
    w = ann['bbox'][2]
    h = ann['bbox'][3]
    ratio = w / h
    ratios.append(ratio)

# 转换为NumPy数组（二维数据，适配K-means输入）
X = np.array(ratios).reshape(-1, 1)

from sklearn.cluster import KMeans

# 设置聚类数量（根据需求调整，如牙齿形态种类）
K = 3

# 运行K-means聚类
kmeans = KMeans(n_clusters=K, random_state=0, n_init=10)  # 显式设置n_init以避免警告
kmeans.fit(X)

# 获取聚类中心并排序
centers = kmeans.cluster_centers_.flatten()
centers.sort()

# 输出聚类中心（宽高比）
print("聚类中心宽高比:", centers)
# 示例输出：[0.31, 0.72, 1.52, 2.98]

# 示例结果处理
anchor_ratios = [round(ratio, 2) for ratio in centers]
print("优化后的锚框比例:", anchor_ratios)
# 输出: [0.3, 0.7, 1.5, 3.0]

In [ ]:
import json
import numpy as np

def calculate_iou(r1, r2):
    """计算两个宽高比的IoU（假设中心点相同且面积相等）"""
    # 假设锚框面积为1，计算宽高
    w1 = np.sqrt(r1)
    h1 = 1 / np.sqrt(r1)
    w2 = np.sqrt(r2)
    h2 = 1 / np.sqrt(r2)
    
    # 计算交叠区域
    inter_w = min(w1, w2)
    inter_h = min(h1, h2)
    intersection = inter_w * inter_h
    
    # 计算联合区域
    union = 1.0 + 1.0 - intersection  # 面积总和-交叠
    
    return intersection / union if union != 0 else 0

class IoUKMeans:
    """基于IoU距离的自定义K-means聚类"""
    def __init__(self, n_clusters=5, max_iter=00, tol=1e-4):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        
    def fit(self, X):
        # 初始化聚类中心
        np.random.seed(0)
        self.centers = X[np.random.choice(X.shape[0], self.n_clusters, replace=False)]
        
        for _ in range(self.max_iter):
            # 分配样本到最近簇（IoU最大）
            distances = np.array([[1 - calculate_iou(x[0], c) 
                                 for c in self.centers.flatten()] 
                                for x in X])
            self.labels_ = np.argmin(distances, axis=1)
            
            # 计算新聚类中心（簇内宽高比中位数）
            new_centers = []
            for i in range(self.n_clusters):
                cluster_samples = X[self.labels_ == i]
                if len(cluster_samples) == 0:
                    new_centers.append(self.centers[i])  # 保持原中心
                else:
                    new_centers.append(np.median(cluster_samples))
            new_centers = np.array(new_centers).reshape(-1, 1)
            
            # 检查收敛条件
            if np.abs(new_centers - self.centers).max() < self.tol:
                break
            self.centers = new_centers
            
        # 排序并保存最终结果
        self.centers = np.sort(self.centers.flatten())
        return self

# 加载标注数据
with open('/data/wangshunyi/mmdetection/data/DENTEX/new_enumeration_data/train.json', 'r') as f:
    data = json.load(f)

# 提取宽高比并转换为numpy数组
ratios = [ann['bbox'][2]/ann['bbox'][3] for ann in data['annotations']]
X = np.array(ratios).reshape(-1, 1)

# 执行IoU优化的K-means聚类
kmeans = IoUKMeans(n_clusters=5)
kmeans.fit(X)

# 处理并输出结果
anchor_ratios = [round(ratio, 2) for ratio in kmeans.centers]
print("IoU优化的锚框比例:", anchor_ratios)  # 示例输出: [0.32, 0.71, 1.49]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

# 指定 COCO 标注文件路径
ann_file = '/data/wangshunyi/mmdetection/data/DENTEX/new_enumeration_data/train.json'  # 替换为你的标注文件路径
coco = COCO(ann_file)

# 获取所有类别和图像 ID
cat_ids = coco.getCatIds()
img_ids = coco.getImgIds()

# 统计所有标注框的宽度和高度
widths = []
heights = []
aspect_ratios = []

# 遍历所有图像
for img_id in img_ids:
    ann_ids = coco.getAnnIds(imgIds=img_id)
    annotations = coco.loadAnns(ann_ids)
    
    for ann in annotations:
        # 提取边界框信息 (x, y, w, h)
        w = ann['bbox'][2]
        h = ann['bbox'][3]
        widths.append(w)
        heights.append(h)
        aspect_ratios.append(w / h)  # 宽高比

# 转换为 NumPy 数组
widths = np.array(widths)
heights = np.array(heights)
aspect_ratios = np.array(aspect_ratios)

# 输出基础统计信息
print(f"总标注框数量: {len(widths)}")
print(f"宽度统计:")
print(f"  Min={np.min(widths):.2f}, Max={np.max(widths):.2f}, Mean={np.mean(widths):.2f}, Std={np.std(widths):.2f}")
print(f"高度统计:")
print(f"  Min={np.min(heights):.2f}, Max={np.max(heights):.2f}, Mean={np.mean(heights):.2f}, Std={np.std(heights):.2f}")
print(f"宽高比统计:")
print(f"  Min={np.min(aspect_ratios):.2f}, Max={np.max(aspect_ratios):.2f}, Mean={np.mean(aspect_ratios):.2f}")

# 绘制分布图
plt.figure(figsize=(15, 5))

# 宽度分布
plt.subplot(1, 3, 1)
plt.hist(widths, bins=100, range=(0, 300), color='blue', alpha=0.7)
plt.title('Width Distribution')
plt.xlabel('Width (pixels)')
plt.ylabel('Count')

# 高度分布
plt.subplot(1, 3, 2)
plt.hist(heights, bins=100, range=(0, 300), color='green', alpha=0.7)
plt.title('Height Distribution')
plt.xlabel('Height (pixels)')

# 宽高比分布
plt.subplot(1, 3, 3)
plt.hist(aspect_ratios, bins=100, range=(0, 3), color='red', alpha=0.7)
plt.title('Aspect Ratio Distribution')
plt.xlabel('Width/Height Ratio')

plt.tight_layout()
plt.show()

In [ ]:
import json
from collections import Counter

def main(json_path, decimal_places=2):
    # 读取JSON文件
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # 提取所有bbox的宽高比
    ratios = []
    for ann in data['annotations']:
        width = ann['bbox'][2]
        height = ann['bbox'][3]
        
        # 计算宽高比并保留指定位数
        if height == 0:  # 避免除零错误
            continue
        ratio = round(width / height, decimal_places)
        ratios.append(ratio)
    
    # 统计频率
    counter = Counter(ratios)
    top_ratios = counter.most_common(50)
    
    # 打印结果
    print(f"最常见的三个宽高比及其出现次数：")
    for ratio, count in top_ratios:
        print(f"宽高比 {ratio} : 出现 {count} 次")


# 使用示例
json_file = "/data/wangshunyi/mmdetection/data/DENTEX/new_enumeration_data/train.json"  # 替换为你的JSON文件路径
main(json_file)

In [ ]:
import json
from collections import Counter

def main(json_path, decimal_places=2):
    # 读取JSON文件
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # 提取类别信息，创建类别ID到类别名称的映射
    category_id_to_name = {category['id']: category['name'] for category in data['categories']}
    
    # 初始化一个字典来存储每个类别的宽高比统计
    category_ratios = {category['name']: [] for category in data['categories']}
    
    # 提取所有bbox的宽高比，并按类别分类
    for ann in data['annotations']:
        width = ann['bbox'][2]
        height = ann['bbox'][3]
        
        # 计算宽高比并保留指定位数
        if height == 0:  # 避免除零错误
            continue
        ratio = round(width / height, decimal_places)
        
        # 获取当前标注的类别名称
        category_id = ann['category_id']
        category_name = category_id_to_name.get(category_id, f"未知类别_{category_id}")
        
        # 将宽高比添加到对应类别的列表中
        category_ratios[category_name].append(ratio)
    
    # 统计每个类别的宽高比频率并找出最常见的
    for category_name, ratios in category_ratios.items():
        if not ratios:
            print(f"类别 '{category_name}' 没有标注信息。")
            continue
        
        counter = Counter(ratios)
        most_common_ratio, most_common_count = counter.most_common(1)[0]
        
        # 打印结果
        print(f"类别 '{category_name}' 最常见的宽高比是 {most_common_ratio}，出现 {most_common_count} 次")

# 使用示例
json_file = "/data/wangshunyi/mmdetection/data/DENTEX/new_enumeration_data/train.json"  # 替换为你的JSON文件路径
main(json_file)

In [ ]:
import json
import numpy as np
from collections import defaultdict

def calculate_area_distribution(json_path, bin_mode='auto', custom_bins=None):
    """
    统计目标框面积分布
    参数：
    - json_path: JSON文件路径
    - bin_mode: 分箱模式，可选 'auto'(自动分箱)/'fixed'(固定步长)/'custom'(自定义区间)
    - custom_bins: 自定义分箱区间，当bin_mode='custom'时生效
    """
    # 读取数据
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # 计算所有面积
    areas = []
    for ann in data['annotations']:
        w, h = ann['bbox'][2], ann['bbox'][3]
        if w > 0 and h > 0:  # 过滤无效标注
            areas.append(w * h)
    
    # 分箱统计
    if bin_mode == 'auto':
        # 自动分箱（基于数据分布）
        hist, bin_edges = np.histogram(areas, bins='auto')
    elif bin_mode == 'fixed':
        # 固定步长分箱（示例：每100像素一个区间）
        max_area = max(areas)
        bin_edges = list(range(0, int(max_area)+100, 100))
        hist, _ = np.histogram(areas, bins=bin_edges)
    elif bin_mode == 'custom' and custom_bins:
        # 自定义分箱
        bin_edges = sorted(custom_bins)
        hist, _ = np.histogram(areas, bins=bin_edges)
    else:
        raise ValueError("Invalid bin mode or missing custom bins")
    
    # 统计结果处理
    area_dist = []
    for i in range(len(hist)):
        lower = bin_edges[i]
        upper = bin_edges[i+1]
        count = hist[i]
        if count > 0:
            area_dist.append( {
                'range': f"{lower}-{upper}",
                'count': int(count),
                'percentage': f"{count/len(areas)*100:.1f}%"
            })
    
    # 附加统计信息
    stats = {
        'total_boxes': len(areas),
        'min_area': min(areas),
        'max_area': max(areas),
        'mean_area': np.mean(areas),
        'median_area': np.median(areas)
    }
    
    return sorted(area_dist, key=lambda x: x['count'], reverse=True), stats

# 使用示例

# 示例1：自动分箱
result, stats = calculate_area_distribution("/data/wangshunyi/mmdetection/data/DENTEX/new_enumeration_data/train.json", bin_mode='auto')

print("\n基础统计信息：")
print(f"目标框总数：{stats['total_boxes']}")
print(f"最小面积：{stats['min_area']:.2f}")
print(f"最大面积：{stats['max_area']:.2f}")
print(f"平均面积：{stats['mean_area']:.2f}")
print(f"中位面积：{stats['median_area']:.2f}")

print("\n常见面积区间分布（自动分箱）：")
print(f"{'区间':<15} | {'数量':<6} | {'占比':<6}")
for item in result[:5]:  # 显示前5大区间
    print(f"{item['range']:<15} | {item['count']:<6} | {item['percentage']:<6}")

# 示例2：自定义分箱
custom_bins = [0, 100, 500, 1000, 5000, 10000]
result, _ = calculate_area_distribution("/data/wangshunyi/mmdetection/data/DENTEX/new_enumeration_data/train.json", 
                                        bin_mode='custom',
                                        custom_bins=custom_bins)

print("\n自定义分箱结果：")
print(f"{'区间':<15} | {'数量':<6} | {'占比':<6}")
for item in result:
    print(f"{item['range']:<15} | {item['count']:<6} | {item['percentage']:<6}")